In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)
from jax_sn.operators.transport_operator import TransportOperatorBatched, TransportOperatorMultipleAnglesVmap, TransportOperatorVmap, TransportOperatorBatchedPallas, TransportOperatorMultipleAngles
import equinox as eqx
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)
from finite_elements.test_moment_scatter_operator import _setup_all_sb

In [2]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain
from jax_sn.source import Source
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory

In [10]:
from finite_elements.test_breeding_blanket import bb_data_dict, base_region_sizes, base_resolutions,  _get_bb_args, _setup_bb_args, _get_bb_args


def setup_mgop(dim, degree):
    MGOP, source_eg0, solution_domain = _setup_bb_args(dim, degree, base_region_sizes(), base_resolutions(), 5.0, 10, 3)
    transport_operator                = TransportOperatorMultipleAnglesVmap(TransportOperatorBatched(MGOP.transport_operator.solution_domain_data, MGOP.transport_operator.topological_sorted_angle, 0, 100))
    MGOP                              = eqx.tree_at(lambda t : t.transport_operator, MGOP, transport_operator)
    return MGOP, source_eg0, solution_domain

MGOP, source_eg0, solution_domain = setup_mgop(2,2)

In [13]:
from jax_sn.tools.memory_analysis import OperatorAnalysis, TotalOperatorSigmaParametrised

In [14]:
onp.random.seed(8012026)
x = onp.random.rand(*MGOP.in_structure().shape)
y = onp.random.rand(*MGOP.out_structure().shape)

print(x.shape)
print(y.shape)
#MGOP.mv(x)

(4, 36, 500, 6)
(4, 36, 500, 6)


In [15]:
MGOP

MultiGroupOperatorScan(
  moment_operator=MomentOperator(
    wk_yklm=f64[72,36],
    quadrature_set=QuadratureSet(angles=f64[72,3], weights=f64[72]),
    elements_basis_shape=ShapeDtypeStruct(shape=(500, 6), dtype=float64)
  ),
  transport_operator=TransportOperatorMultipleAnglesVmap(
    transport_operator=TransportOperatorBatched(
      solution_domain_data=SolutionDomainData(
        domain=Domain(
          vertices=f64[286,2],
          connectivity=i64[500,3],
          face_normals=f64[500,3,3],
          etoe=weak_i64[500,3],
          etof=weak_i64[500,3],
          face_perm_idx=i64[500,3],
          face_template=i64[3,2],
          cross_sections_indexed=CrossSectionIndexed(
            indices=i64[500],
            cross_section_data=CrossSectionData(
              total=f64[8,4], scattering=f64[8,6,4,4]
            )
          )
        ),
        fem_matrices=FEMMatrices(
          mass_matrix=f64[6,6],
          integration_coeff=f64[6],
          derivative_matrix=f64

In [16]:
def _pprint_memory_analysis(memory_dict):
    # structure is like:
    # {'peak': 215.2952651977539, 'argument': 104.98715209960938, 'output': 82.3974838256836, 'temp': 27.91297149658203}
    print("peak           :", memory_dict["peak"] , " MB")
    print("argument       :", memory_dict["argument"] , " MB")
    print("output         :", memory_dict["output"] , " MB")
    print("temp           :", memory_dict["temp"] , " MB")

In [17]:

def _pprint_memory_analysis(memory_dict):
    # structure is like:
    # {'peak': 215.2952651977539, 'argument': 104.98715209960938, 'output': 82.3974838256836, 'temp': 27.91297149658203}
    print("peak           :", memory_dict["peak"] , " MB")
    print("argument       :", memory_dict["argument"] , " MB")
    print("output         :", memory_dict["output"] , " MB")
    print("temp           :", memory_dict["temp"] , " MB")
    
def analyse_operators(MGOP):
    print("TransportOperator")
    _pprint_memory_analysis(OperatorAnalysis.analyze_memory(MGOP.transport_operator, "inv"))
    print("MassOperator")
    _pprint_memory_analysis(OperatorAnalysis.analyze_memory(MGOP.mass_operator, "mv"))
    print("MomentOperator")
    _pprint_memory_analysis(OperatorAnalysis.analyze_memory(MGOP.moment_operator, "mv"))
    print("ScatterOperator")
    _pprint_memory_analysis(OperatorAnalysis.analyze_memory(MGOP.scatter_operator, "mv"))
    print("Single Group Operator")
    sgop = MGOP.to_single_group_operator(0)
    _pprint_memory_analysis(OperatorAnalysis.analyze_memory(sgop, "mv"))
    print("Multi Group Operator")
    _pprint_memory_analysis(OperatorAnalysis.analyze_memory(MGOP, "mv"))

print("Memory Analysis for MV of Operators")
analyse_operators(MGOP)

Memory Analysis for MV of Operators
TransportOperator
peak           : 12.206832885742188  MB
argument       : 2.34576416015625  MB
output         : 1.6479721069335938  MB
temp           : 8.215461730957031  MB
MassOperator
peak           : 6.59619140625  MB
argument       : 1.65203857421875  MB
output         : 1.6479721069335938  MB
temp           : 3.296875  MB
MomentOperator
peak           : 4.159454345703125  MB
argument       : 1.667724609375  MB
output         : 0.8239974975585938  MB
temp           : 1.668212890625  MB
ScatterOperator
peak           : 6.6411895751953125  MB
argument       : 3.325347900390625  MB
output         : 1.6479721069335938  MB
temp           : 1.6679840087890625  MB
Single Group Operator
peak           : 13.743438720703125  MB
argument       : 1.575103759765625  MB
output         : 0.8239974975585938  MB
temp           : 11.347297668457031  MB
Multi Group Operator
peak           : 18.869163513183594  MB
argument       : 4.047027587890625  MB
output     

In [18]:
rhs_eg0 = MGOP.moment_operator.mv(MGOP.transport_operator.to_energy_group(0).inv(MGOP.mass_operator.mv(source_eg0)))
rhs_total = jnp.stack([rhs_eg0] + [jnp.zeros_like(rhs_eg0) for _ in range(MGOP.n_energy_groups -1)], axis =0)

In [19]:
from jax_sn.operators.multi_group_operator.multi_group_operator_imp import solve_lower_triangular, solve_total

import lineax as lx
bicgstab_solver = lx.BiCGStab(1e-10, 1e-10, max_steps = 5)

@eqx.filter_jit
def solve(sgop, rhs):
    return lx.linear_solve(sgop, rhs, solver = bicgstab_solver, throw = False).value

def analyse_solve_memory(MGOP):

    print("Single Group Operator Solve")
    sgop = MGOP.to_single_group_operator(0)
    _pprint_memory_analysis(_filter_jit_f_to_memory(solve, sgop, rhs_total[0]))
    print("Multi Group Operator Total Solve")
    _pprint_memory_analysis(_filter_jit_f_to_memory(solve, MGOP, rhs_total))
    print("Multi Group Operator Group-wise Solve")
    _pprint_memory_analysis(_filter_jit_f_to_memory(solve_lower_triangular, MGOP, bicgstab_solver, rhs_total))
    
analyse_solve_memory(MGOP)
    

Single Group Operator Solve
peak           : 19.040367126464844  MB
argument       : 1.6258697509765625  MB
output         : 0.8239974975585938  MB
temp           : 17.1259765625  MB
Multi Group Operator Total Solve
peak           : 38.678627014160156  MB
argument       : 4.0977935791015625  MB
output         : 3.2959213256835938  MB
temp           : 32.310546875  MB
Multi Group Operator Group-wise Solve
peak           : 24.8082275390625  MB
argument       : 4.0977935791015625  MB
output         : 3.2959213256835938  MB
temp           : 17.42725372314453  MB


In [21]:
def _pprint_timing_analysis(timing_dict):
    print(f"\tmv time      : {timing_dict      * 1000:.2f} ms")

def analyse_timings(MGOP):
    print("TransportOperator")
    _pprint_timing_analysis(OperatorAnalysis.analyze_timing(MGOP.transport_operator, "inv", n_samples=3, ignore_n=1))
    print("MassOperator")
    _pprint_timing_analysis(OperatorAnalysis.analyze_timing(MGOP.mass_operator, "mv", n_samples=3, ignore_n=1))
    print("MomentOperator")
    _pprint_timing_analysis(OperatorAnalysis.analyze_timing(MGOP.moment_operator, "mv", n_samples=3, ignore_n=1))
    print("ScatterOperator")
    _pprint_timing_analysis(OperatorAnalysis.analyze_timing(MGOP.scatter_operator, "mv", n_samples=3, ignore_n=1))
    print("Single Group Operator")
    sgop = MGOP.to_single_group_operator(0)
    _pprint_timing_analysis(OperatorAnalysis.analyze_timing(sgop, "mv", n_samples=3, ignore_n=1))
    print("Multi Group Operator")
    _pprint_timing_analysis(OperatorAnalysis.analyze_timing(MGOP, "mv", n_samples=3, ignore_n=1))

print("Timing Analysis for MV of Operators")
analyse_timings(MGOP)

Timing Analysis for MV of Operators
TransportOperator
	mv time      : 84.70 ms
MassOperator
	mv time      : 0.73 ms
MomentOperator
	mv time      : 0.58 ms
ScatterOperator
	mv time      : 0.54 ms
Single Group Operator
	mv time      : 22.06 ms
Multi Group Operator
	mv time      : 77.83 ms


In [27]:
bicgstab_solver = lx.BiCGStab(1e-10, 1e-10, max_steps = 5)

def solve_lower_triangular_with_stats(mgop, solver, rhs : jnp.ndarray) -> jnp.ndarray:
    '''
    Solve the linear system using a given solver, assuming the multi-group operator is lower triangular.
    This is useful for e.g. downscattering only problems. 

    Parameters
    ----------
    mgop : MultiGroupOperator
        The multi group operator.
    solver : lx.LinearSolver
        The linear solver to use.
    rhs : jnp.ndarray
        The right-hand side.
    Returns
    -------
    jnp.ndarray
        The solution.   
    '''
    n_groups = mgop.n_energy_groups    
    solution = jnp.zeros_like(rhs)
    stats = [] 

    for i in range(n_groups):
        
        sgop = mgop.to_single_group_operator(i)
        rhs_i = rhs[i] + mgop.moment_operator.mv(sgop.transport_operator.inv(mgop.mass_operator.mv(mgop.scatter_operator.to_energy_group(i).mv(solution))))        
        solution_i = lx.linear_solve(sgop, rhs_i, solver, throw = False)
        solution = solution.at[i].set(solution_i.value)
        print(solution_i.stats)
        
        stats.append(solution_i.stats)
    return solution, stats

solution, stats = solve_lower_triangular_with_stats(MGOP, bicgstab_solver, rhs_total)
# I know it is not going to converge in 5, so I can just /5 / 2 to get the time for a single MV expected

{'max_steps': 5, 'num_steps': Array(5, dtype=int64)}
{'max_steps': 5, 'num_steps': Array(5, dtype=int64)}
{'max_steps': 5, 'num_steps': Array(5, dtype=int64)}
{'max_steps': 5, 'num_steps': Array(5, dtype=int64)}


# I know it is not going to converge in 5, so 

In [30]:
from jax_sn.tools.memory_analysis import custom_timeit
def analyse_solve_timing(mgop):
    def solve_lower_triangular_part():
        return solve_lower_triangular(mgop, bicgstab_solver, rhs_total)
    time, std_time = custom_timeit(solve_lower_triangular_part, number = 3)
    
    print(f"Lower triangular solve time: {time     * 1000:.2f} ms ")
    time_d_n_mv = time / (mgop.n_energy_groups * bicgstab_solver.max_steps * 2)
    print(f"\t Total time divided by number of single group MV: {time_d_n_mv * 1000:.2f} ms ")

analyse_solve_timing(MGOP)


Lower triangular solve time: 955.80 ms 
	 Total time divided by number of single group MV: 23.89 ms 
